In [4]:
import pandas as pd
import numpy as np
pd.set_option('display.width', 140)

ev = pd.read_csv('dataset_task/evaluations.csv.gz', low_memory=False)
co = pd.read_csv('dataset_task/car_outcomes.csv')
mk = pd.read_csv('dataset_task/market_kolesa.csv.gz', low_memory=False)
ti = pd.read_csv('dataset_task/tech_inspection.csv.gz', low_memory=False)
st = pd.read_csv('dataset_task/stage_times.csv.gz', low_memory=False)
pp = pd.read_csv('dataset_task/pricing_policy.csv')

for name, df in [('evaluations', ev), ('car_outcomes', co), ('market_kolesa', mk),
                 ('tech_inspection', ti), ('stage_times', st), ('pricing_policy', pp)]:
    print(f'{name:16s} shape={df.shape}')

evaluations      shape=(164400, 86)
car_outcomes     shape=(27426, 42)
market_kolesa    shape=(1137602, 19)
tech_inspection  shape=(1497772, 11)
stage_times      shape=(176674, 53)
pricing_policy   shape=(8, 14)


In [5]:
chain_cols = ['first_pricer_buy_price','second_pricer_buy_price','confirmed_pricer_buy_price',
              'confirmed_rgv_buy_price','rov_buy_price','head_branch_buy_price']
purchased = ev['purchase_date'].notna()
rows = []
for c in chain_cols:
    rows.append({
        'field': c,
        'filled_if_purchased': ev.loc[purchased, c].notna().mean(),
        'filled_if_not_purchased': ev.loc[~purchased, c].notna().mean(),
    })
print(pd.DataFrame(rows).to_string(index=False))
print()
print('second_pricer_, confirmed_, rov_, head_branch_, first_pricer_buy_price исключены из признаков.')

                     field  filled_if_purchased  filled_if_not_purchased
    first_pricer_buy_price             0.997847                 0.984957
   second_pricer_buy_price             0.998046                 0.084523
confirmed_pricer_buy_price             0.997284                 0.084471
   confirmed_rgv_buy_price             0.991752                 0.012056
             rov_buy_price             0.997847                 0.011996
     head_branch_buy_price             0.997913                 0.010707

second_pricer_, confirmed_, rov_, head_branch_, first_pricer_buy_price исключены из признаков.


In [6]:
fully_empty = [c for c in ev.columns if ev[c].isna().all()]
print('пустые колонки:', fully_empty)

пустые колонки: ['is_approved_by_rop', 'is_turbo', 'is_gas', 'seats_count', 'is_pledge']


In [7]:
print('evaluation_date:', ev['evaluation_date'].min(), '->', ev['evaluation_date'].max())
print('tech_inspection created_date:', ti['created_date'].min(), '->', ti['created_date'].max())

print('тех инспекция только за 27.07-07.09.2026')

evaluation_date: 2024-09-01 09:32:46.982 -> 2026-09-07 20:31:36.030
tech_inspection created_date: 2026-07-27 00:00:03.571 -> 2026-09-07 20:37:49.978
тех инспекция только за 27.07-07.09.2026


In [8]:
# квантили у колес почти все пустые

In [9]:
for c in ['avg_price','median_price_w','q1_price','q3_price','stdev_price_avg']:
    print(f'{c:18s} заполнено {mk[c].notna().mean():.1%}')

print('ads_cnt:', mk['ads_cnt'].median())

avg_price          заполнено 100.0%
median_price_w     заполнено 2.4%
q1_price           заполнено 2.4%
q3_price           заполнено 2.4%
stdev_price_avg    заполнено 29.7%
ads_cnt: 2.0


In [10]:
print(ev['total_car_state'].value_counts(dropna=False)) # 0/nan значения в оценке качества авто

total_car_state
Среднее     77104
Хорошее     43047
Плохое      25927
Отличное    10450
Ужасное      6306
NaN          1563
0               3
Name: count, dtype: int64


In [11]:
print(co['status'].value_counts(dropna=False))
print('доля gm2 < 0 (отрицательная маржа):', (co['gm2'] < 0).mean())
print('диапазон дат покупок:', co['date_buy'].min(), '->', co['date_buy'].max())

status
продан      26385
на стоке     1041
Name: count, dtype: int64
доля gm2 < 0 (отрицательная маржа): 0.11335958579450157
диапазон дат покупок: 2024-09-01 11:13:40 -> 2026-09-07 21:00:25


In [12]:
purchased_ev = ev[ev['purchase_date'].notna()].copy()
purchased_ev['matched'] = purchased_ev['deal_key'].isin(co['deal_key'])
print('всего выкупов:', len(purchased_ev), ' | в car_outcomes:', purchased_ev['matched'].sum(),
      f"({purchased_ev['matched'].mean():.1%})")
print(purchased_ev.groupby('matched')['transaction_group'].value_counts(normalize=True).unstack().round(3).to_string())

print('признаки для обучения purchase/trade_in/trade_up')

всего выкупов: 30188  | в car_outcomes: 21787 (72.2%)
transaction_group  back_purch  commission   damu  overauto  partners  purchase  trade_in  trade_up
matched                                                                                           
False                   0.016       0.529  0.003      0.00     0.032     0.274     0.076     0.069
True                    0.026       0.087  0.005      0.01     0.072     0.472     0.161     0.166
признаки для обучения purchase/trade_in/trade_up


In [13]:
import sys; sys.path.insert(0, '../src') #воронка
from funnel import STAGE_COLUMNS, funnel_table, total_duration_by_outcome

print('evaluations:', len(ev), ' stage_times:', len(st),
      ' пересечение по evaluation_id:', len(set(ev.evaluation_id) & set(st.evaluation_id)))
print()
ft = funnel_table(st)
print(ft.to_string(index=False))
print()
print(total_duration_by_outcome(ev, st).to_string(index=False))

evaluations: 164400  stage_times: 176674  пересечение по evaluation_id: 163878

                          stage  reached  reached_share  median_minutes  p95_minutes
             initial_inspection   176494       0.998981            24.0        50.00
on_complete_for_init_inspection    24076       0.136274             6.0        46.00
           first_check_eval_rgv       37       0.000209             4.0        79.00
       initial_evaluation_queue   172490       0.976318             6.0        24.00
             initial_evaluation   172470       0.976205             2.0         8.00
             first_eval_approve    32726       0.185234            13.0       148.00
    rework_first_check_eval_rgv       14       0.000079             4.5        74.85
   initial_evaluation_completed   172753       0.977807            23.0     13197.00
                client_thinking   127415       0.721187         13366.0     20591.00
                   queue_for_mp    50206       0.284173             6.